# ICC-to-sRGB metadata-safe export verification — v3

**Provisional labels; non-clinical; non-final; participant independence unverified.**

Generated-profile/colour-management checks only. No dental images, Test data, Drive, model operations, installs or weights. Reuse completed historical workflows; do not repeat architecture/trainer tests.

Upload ONLY `vision_pp1_icc_safe_code_v3.zip`. Acceptance: 95 selected checks passed, no failures/errors/skips, seven unchanged source hashes, LittleCMS available. The new notebook has not executed in Colab. Keep RUN_PILOT=False; no real-data cell exists. Supported ICC is converted before resizing; invalid/ambiguous profiles or excessive round-trip error stop export. Actual dental pixel conversions are not certified by these tests.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, json, os, stat, sys, zipfile

NOTICE = 'Provisional labels; non-clinical; non-final; participant independence unverified.'
print(NOTICE)
ROOT = Path('/content/pp1_icc_safe_export_v3')
CODE_ROOT = ROOT / 'code'
CODE_ARCHIVE = 'vision_pp1_icc_safe_code_v3.zip'
CODE_SHA256 = '324f9fe84a4783b71ccc1371a1764ffb3665f8670cc6cc97377d005990acf7d6'
EXPECTED_SOURCES = ['src/vision/model_pipeline.py', 'src/vision/train_pipeline.py', 'src/vision/exploratory_bundle.py', 'src/vision/exploratory_pilot.py', 'tests/test_exploratory_bundle.py', 'tests/test_exploratory_pilot.py', 'tests/test_icc_colour_export.py']
if Path('/content/drive').exists() or CODE_ROOT.exists():
    raise RuntimeError('Use a fresh runtime without Drive; preserve any existing files.')
uploaded = files.upload()
if set(uploaded) != {CODE_ARCHIVE}:
    raise RuntimeError('Upload only the code-only ZIP; no dataset archives.')
if hashlib.sha256(uploaded[CODE_ARCHIVE]).hexdigest() != CODE_SHA256:
    raise RuntimeError('Code archive SHA-256 mismatch')
with zipfile.ZipFile(CODE_ARCHIVE) as archive:
    entries = archive.infolist()
    expected = set(EXPECTED_SOURCES) | {'code_manifest.json'}
    if len(entries) != len(expected) or {e.filename for e in entries} != expected:
        raise RuntimeError('Unexpected code archive contents')
    if sum(e.file_size for e in entries) > 16 * 1024**2:
        raise RuntimeError('Code archive size limit')
    for entry in entries:
        p = PurePosixPath(entry.filename)
        if p.is_absolute() or '..' in p.parts or stat.S_IFMT(entry.external_attr >> 16) not in {0, stat.S_IFREG}:
            raise RuntimeError('Unsafe code entry')
    code_manifest = json.loads(archive.read('code_manifest.json'))
    if set(code_manifest['files']) != set(EXPECTED_SOURCES) or code_manifest['dental_data_included'] is not False:
        raise RuntimeError('Code manifest mismatch')
    CODE_ROOT.mkdir(parents=True)
    for entry in entries:
        target = CODE_ROOT / entry.filename
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(entry))
for name, digest in code_manifest['files'].items():
    if hashlib.sha256((CODE_ROOT / name).read_bytes()).hexdigest() != digest:
        raise RuntimeError('Source hash mismatch: ' + name)
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print(NOTICE, '\nAll seven source hashes verified. No dental data uploaded.')


In [ ]:
import importlib.util
required = ['numpy', 'PIL', 'pytest']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing dependencies; no automatic install: ' + str(missing))
import numpy, PIL, pytest
from PIL import ImageCms, features
if not features.check('littlecms2'):
    raise RuntimeError('Pillow LittleCMS is unavailable; do not guess source colours or proceed.')
from src.vision import exploratory_bundle as eb
runtime = {'python': sys.version, 'numpy': numpy.__version__, 'Pillow': PIL.__version__,
           'pytest': pytest.__version__, 'littlecms2': features.version('littlecms2'),
           'device': 'CPU; no model operations'}
RUN_PILOT = False
print(json.dumps(runtime, indent=2))
print('Synthetic ICC conversion only; no dental data, model or weights loaded.')


In [ ]:
import subprocess
import xml.etree.ElementTree as ET

print(NOTICE)
SELECTED_TESTS = ['tests/test_icc_colour_export.py', 'tests/test_exploratory_bundle.py', 'tests/test_exploratory_pilot.py::test_cam_artifacts_always_carry_notice_and_unsupported_prediction', 'tests/test_exploratory_pilot.py::test_unlisted_cam_example_blocked_before_image_read', 'tests/test_exploratory_pilot.py::test_cam_rejects_payload_tampering_before_explanation', 'tests/test_exploratory_pilot.py::test_production_training_gate_still_rejects_exploratory_metadata']
environment = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD='1', PYTHONDONTWRITEBYTECODE='1')
completed = subprocess.run([sys.executable, '-B', '-m', 'pytest', '-q', '-rs', '--tb=short',
                           '-p', 'no:cacheprovider', '--junitxml=icc_export_checks.xml', *SELECTED_TESTS],
                          env=environment, capture_output=True, text=True)
log = NOTICE + '\n' + completed.stdout + completed.stderr
Path('icc_export_pytest.log').write_text(log)
print(log)
xml = ET.parse('icc_export_checks.xml').getroot()
suites = list(xml) if xml.tag == 'testsuites' else [xml]
counts = {key: sum(int(s.get(key, 0)) for s in suites) for key in ['tests', 'failures', 'errors', 'skipped']}
counts['passed'] = counts['tests'] - counts['failures'] - counts['errors'] - counts['skipped']
after_hashes = {name: eb.sha256(CODE_ROOT / name) for name in EXPECTED_SOURCES}
if after_hashes != code_manifest['files']:
    raise RuntimeError('Source files changed during verification')
verification = {'notice': NOTICE, 'mode': 'synthetic_ICC_export_only', 'counts': counts,
                'runtime': runtime, 'source_hashes_before': code_manifest['files'], 'source_hashes_after': after_hashes,
                'code_archive_sha256': CODE_SHA256, 'RUN_PILOT': RUN_PILOT,
                'pytest_returncode': completed.returncode, 'dental_images_exported': 0,
                'dental_training_run': False, 'test_data_used': False}
Path('icc_export_runtime_verification.json').write_text(json.dumps(verification, indent=2))
print(json.dumps(verification, indent=2))
if completed.returncode != 0 or counts != {'tests': 95, 'failures': 0, 'errors': 0, 'skipped': 0, 'passed': 95}:
    raise RuntimeError('Metadata-safe regression verification incomplete; no real packaging.')


In [ ]:
print(NOTICE)
evidence = CODE_ROOT / 'vision_icc_colour_runtime_evidence_v3.zip'
if evidence.exists():
    raise RuntimeError('Evidence ZIP already exists; preserve it.')
with zipfile.ZipFile(evidence, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ['icc_export_runtime_verification.json', 'icc_export_pytest.log', 'icc_export_checks.xml', 'code_manifest.json']:
        archive.write(CODE_ROOT / name, name)
files.download(str(evidence))


In [ ]:
assert RUN_PILOT is False
print('STOP: no dental data packaged/uploaded; no training or inference authorised.')
